# Model Diagnostics – Feature Importance & Leakage Detection

**Purpose:** Diagnose lift chart patterns and model behavior

**Flexible Configuration:**
- Analyze ONE model (fast iteration during tuning)
- Analyze ALL FOUR models (comprehensive comparison)

**Key Questions:**
1. Is there data leakage (features revealing claim occurrence)?
2. What's the distribution of zeros vs non-zeros?
3. Which features have the most importance?
4. Are predictions binary (near-0 or high) vs continuous?
5. Is the decile pattern expected for zero-inflated insurance data?

---
## 0 · Setup & Configuration

In [ ]:
import os, sys
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "code"))

# Set DEBUG to match training (2 = 10% sample)
DEBUG = 2

print(f"Project root: {PROJECT_ROOT}")
print(f"DEBUG mode: {DEBUG} (10% sample)")

---
## 1 · Load Saved Models

In [ ]:
from model_training import load_experiment

# Load all 4 models
model1 = load_experiment("type1_ordinal")
model2 = load_experiment("type2_binary")
model3 = load_experiment("type3_actuarial")
model4 = load_experiment("type4_custom")

print("\n" + "="*70)
print("MODEL 1: Type 1 Ordinal")
print("="*70)
print(f"  Features: {model1['metrics']['n_features']}")
print(f"  Train RMSE: {model1['metrics'].get('train_rmse', 'N/A')}")
print(f"  Train MAE: {model1['metrics'].get('train_mae', 'N/A')}")
print(f"  Train R²: {model1['metrics'].get('train_r2', 'N/A')}")

print("\n" + "="*70)
print("MODEL 2: Type 2 Binary")
print("="*70)
print(f"  Features: {model2['metrics']['n_features']}")
print(f"  Train RMSE: {model2['metrics'].get('train_rmse', 'N/A')}")
print(f"  Train MAE: {model2['metrics'].get('train_mae', 'N/A')}")
print(f"  Train R²: {model2['metrics'].get('train_r2', 'N/A')}")

print("\n" + "="*70)
print("MODEL 3: Type 3 Actuarial")
print("="*70)
print(f"  Features: {model3['metrics']['n_features']}")
print(f"  Train RMSE: {model3['metrics'].get('train_rmse', 'N/A')}")
print(f"  Train MAE: {model3['metrics'].get('train_mae', 'N/A')}")
print(f"  Train R²: {model3['metrics'].get('train_r2', 'N/A')}")

print("\n" + "="*70)
print("MODEL 4: Type 4 Custom")
print("="*70)
print(f"  Features: {model4['metrics']['n_features']}")
print(f"  Train RMSE: {model4['metrics'].get('train_rmse', 'N/A')}")
print(f"  Train MAE: {model4['metrics'].get('train_mae', 'N/A')}")
print(f"  Train R²: {model4['metrics'].get('train_r2', 'N/A')}")

---
## 2 · Load Training Data (Same as Training)

In [ ]:
from encoding_strategies import load_train_only, get_y

# Load same 10% sample used in training
train_raw = load_train_only(debug=DEBUG)
y_train = get_y(train_raw)

print(f"\nTraining data shape: {train_raw.shape}")
print(f"Target variable (y_train) shape: {y_train.shape}")

---
## 3 · Target Variable Analysis

**Key Question:** Is the target mostly zeros? How many extreme outliers?

In [ ]:
# Zero vs non-zero breakdown
n_zeros = (y_train == 0).sum()
n_nonzeros = (y_train > 0).sum()
pct_zeros = 100 * n_zeros / len(y_train)

print("="*70)
print("TARGET VARIABLE DISTRIBUTION")
print("="*70)
print(f"Total records:    {len(y_train):,}")
print(f"Zeros:            {n_zeros:,} ({pct_zeros:.2f}%)")
print(f"Non-zeros:        {n_nonzeros:,} ({100-pct_zeros:.2f}%)")
print(f"\nMean:             ${y_train.mean():.2f}")
print(f"Median:           ${y_train.median():.2f}")
print(f"Std:              ${y_train.std():.2f}")
print(f"\nPercentiles:")
print(f"  50th (median):  ${y_train.quantile(0.50):.2f}")
print(f"  75th:           ${y_train.quantile(0.75):.2f}")
print(f"  90th:           ${y_train.quantile(0.90):.2f}")
print(f"  95th:           ${y_train.quantile(0.95):.2f}")
print(f"  99th:           ${y_train.quantile(0.99):.2f}")
print(f"  Max:            ${y_train.max():.2f}")

# Check ceiling cap
TARGET_CEILING = 100000
n_capped = (y_train >= TARGET_CEILING).sum()
print(f"\nCapped at ${TARGET_CEILING:,}: {n_capped:,} ({100*n_capped/len(y_train):.2f}%)")

# Histogram
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Full distribution
axes[0].hist(y_train, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Pure Premium ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Distribution (All Values)')
axes[0].grid(alpha=0.3)

# Non-zero only
y_nonzero = y_train[y_train > 0]
axes[1].hist(y_nonzero, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Pure Premium ($)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Target Distribution (Non-Zeros Only, n={len(y_nonzero):,})')
axes[1].grid(alpha=0.3)

# Log scale (non-zeros)
axes[2].hist(np.log10(y_nonzero + 1), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[2].set_xlabel('log10(Pure Premium + 1)')
axes[2].set_ylabel('Count')
axes[2].set_title('Target Distribution (Log Scale, Non-Zeros)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n⚠️  HIGH % OF ZEROS IS EXPECTED in insurance data (most policies have no claims)")
print("⚠️  But if model perfectly separates zeros from non-zeros, check for LEAKAGE!")

---
## 4 · Feature Importance – LEAKAGE DETECTION

**Goal:** Identify if any feature is "too perfect" at predicting claims

In [ ]:
def analyze_feature_importance(model_dict, model_name, top_n=20):
    """
    Analyze feature importance and flag potential leakers.
    """
    model = model_dict['model']
    feature_names = model_dict['feature_names']
    
    # Get importance
    importance = model.feature_importances_
    feat_imp_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    print("\n" + "="*70)
    print(f"{model_name} - TOP {top_n} FEATURES BY IMPORTANCE")
    print("="*70)
    
    top_features = feat_imp_df.head(top_n)
    
    # Flag suspicious features
    leakage_keywords = ['claim', 'loss', 'pp_', 'ee_', 'exposure', 'premium']
    top_features['⚠️_LEAKAGE_RISK'] = top_features['feature'].apply(
        lambda x: any(keyword in x.lower() for keyword in leakage_keywords)
    )
    
    # Check for dominant feature (>50% importance)
    max_importance = top_features['importance'].iloc[0]
    if max_importance > 0.5:
        print(f"\n🚨 WARNING: Top feature has {max_importance*100:.1f}% importance - POSSIBLE LEAKAGE!")
    
    # Display
    print(top_features.to_string(index=False))
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))
    colors = ['red' if leak else 'steelblue' for leak in top_features['⚠️_LEAKAGE_RISK']]
    ax.barh(range(top_n), top_features['importance'].values[:top_n], color=colors)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top_features['feature'].values[:top_n])
    ax.set_xlabel('Feature Importance (Gain)')
    ax.set_title(f'{model_name} - Top {top_n} Features (🔴 = Leakage Risk)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    return feat_imp_df

# Analyze all 4 models
imp1 = analyze_feature_importance(model1, "MODEL 1: Type 1 Ordinal", top_n=20)
imp2 = analyze_feature_importance(model2, "MODEL 2: Type 2 Binary", top_n=20)
imp3 = analyze_feature_importance(model3, "MODEL 3: Type 3 Actuarial", top_n=20)
imp4 = analyze_feature_importance(model4, "MODEL 4: Type 4 Custom", top_n=20)

---
## 5 · Prediction Analysis

**Key Question:** Are predictions binary (near-0 or high) or continuous?

In [ ]:
from encoding_strategies import (
    encode_type1_ordinal,
    encode_type2_binary,
    encode_type3_actuarial,
    encode_type4_custom
)

# Encode training data for all 4 models
print("Encoding training data...")
X_train_m1, _, _ = encode_type1_ordinal(train_raw)
X_train_m2, _, _ = encode_type2_binary(train_raw)
X_train_m3, _, _ = encode_type3_actuarial(train_raw)
X_train_m4, _, _ = encode_type4_custom(train_raw)

# ✅ ALIGN FEATURES TO SAVED MODEL (critical for OHE columns!)
print("Aligning features to saved models...")

# Model 1: Align to saved feature names
saved_features_m1 = model1['feature_names']
missing_cols_m1 = [c for c in saved_features_m1 if c not in X_train_m1.columns]
if missing_cols_m1:
    print(f"  Model 1: Adding {len(missing_cols_m1)} missing columns (filling with 0)")
    for col in missing_cols_m1:
        X_train_m1[col] = 0
X_train_m1 = X_train_m1[saved_features_m1]  # Reorder to match saved order

# Model 2: Align to saved feature names
saved_features_m2 = model2['feature_names']
missing_cols_m2 = [c for c in saved_features_m2 if c not in X_train_m2.columns]
if missing_cols_m2:
    print(f"  Model 2: Adding {len(missing_cols_m2)} missing columns (filling with 0)")
    for col in missing_cols_m2:
        X_train_m2[col] = 0
X_train_m2 = X_train_m2[saved_features_m2]  # Reorder to match saved order

# Model 3: Align to saved feature names
saved_features_m3 = model3['feature_names']
missing_cols_m3 = [c for c in saved_features_m3 if c not in X_train_m3.columns]
if missing_cols_m3:
    print(f"  Model 3: Adding {len(missing_cols_m3)} missing columns (filling with 0)")
    for col in missing_cols_m3:
        X_train_m3[col] = 0
X_train_m3 = X_train_m3[saved_features_m3]  # Reorder to match saved order

# Model 4: Align to saved feature names
saved_features_m4 = model4['feature_names']
missing_cols_m4 = [c for c in saved_features_m4 if c not in X_train_m4.columns]
if missing_cols_m4:
    print(f"  Model 4: Adding {len(missing_cols_m4)} missing columns (filling with 0)")
    for col in missing_cols_m4:
        X_train_m4[col] = 0
X_train_m4 = X_train_m4[saved_features_m4]  # Reorder to match saved order

print("✅ Feature alignment complete!")

# Generate predictions for all 4 models
X_train_m1_filled = X_train_m1.fillna(-1)
X_train_m2_filled = X_train_m2.fillna(-1)
X_train_m3_filled = X_train_m3.fillna(-1)
X_train_m4_filled = X_train_m4.fillna(-1)

preds_m1 = model1['model'].predict(X_train_m1_filled)
preds_m2 = model2['model'].predict(X_train_m2_filled)
preds_m3 = model3['model'].predict(X_train_m3_filled)
preds_m4 = model4['model'].predict(X_train_m4_filled)

print("\n" + "="*70)
print("PREDICTION STATISTICS")
print("="*70)

for model_name, preds in [("Model 1 (Ordinal)", preds_m1), 
                          ("Model 2 (Binary)", preds_m2),
                          ("Model 3 (Actuarial)", preds_m3),
                          ("Model 4 (Custom)", preds_m4)]:
    print(f"\n{model_name}:")
    print(f"  Mean:     ${preds.mean():.2f}")
    print(f"  Median:   ${np.median(preds):.2f}")
    print(f"  Std:      ${preds.std():.2f}")
    print(f"  Min:      ${preds.min():.2f}")
    print(f"  Max:      ${preds.max():.2f}")
    print(f"  Unique values: {len(np.unique(preds)):,}")
    
    # Check for binary-like predictions
    n_near_zero = (preds < 10).sum()
    pct_near_zero = 100 * n_near_zero / len(preds)
    print(f"  Near-zero (<$10): {n_near_zero:,} ({pct_near_zero:.2f}%)")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Model 1 histogram
axes[0,0].hist(preds_m1, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0,0].set_xlabel('Predicted Pure Premium ($)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Model 1 (Ordinal) - Prediction Distribution')
axes[0,0].grid(alpha=0.3)

# Model 1 scatter
axes[0,1].scatter(preds_m1, y_train, alpha=0.3, s=1, color='steelblue')
axes[0,1].plot([0, max(preds_m1.max(), y_train.max())], 
               [0, max(preds_m1.max(), y_train.max())], 
               'r--', linewidth=2, label='Perfect prediction')
axes[0,1].set_xlabel('Predicted ($)')
axes[0,1].set_ylabel('Actual ($)')
axes[0,1].set_title('Model 1 (Ordinal) - Predicted vs Actual')
axes[0,1].legend()
axes[0,1].grid(alpha=0.3)

# Model 2 histogram
axes[1,0].hist(preds_m2, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1,0].set_xlabel('Predicted Pure Premium ($)')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Model 2 (Actuarial) - Prediction Distribution')
axes[1,0].grid(alpha=0.3)

# Model 2 scatter
axes[1,1].scatter(preds_m2, y_train, alpha=0.3, s=1, color='orange')
axes[1,1].plot([0, max(preds_m2.max(), y_train.max())], 
               [0, max(preds_m2.max(), y_train.max())], 
               'r--', linewidth=2, label='Perfect prediction')
axes[1,1].set_xlabel('Predicted ($)')
axes[1,1].set_ylabel('Actual ($)')
axes[1,1].set_title('Model 2 (Actuarial) - Predicted vs Actual')
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n⚠️  If predictions are clustered at 2 values (near-0 and high), this suggests leakage!")
print("✅  Good predictions should be continuous across a range.")

---
## 6 · Decile Breakdown – Zero vs Non-Zero

**Key Question:** Are deciles 1-8 ALL zeros and decile 9-10 ALL non-zeros?

In [ ]:
def analyze_deciles(y_true, y_pred, model_name, bins=10):
    """
    Analyze composition of each decile (zeros vs non-zeros).
    """
    df = pd.DataFrame({
        'y_true': y_true,
        'y_pred': y_pred,
    })
    
    # Sort by predicted (ascending) and assign deciles
    df = df.sort_values('y_pred', ascending=True).reset_index(drop=True)
    df['decile'] = pd.qcut(df.index, bins, labels=False, duplicates='drop') + 1
    
    # Add zero indicator
    df['is_zero'] = (df['y_true'] == 0).astype(int)
    
    # Group by decile
    decile_stats = df.groupby('decile').agg(
        count=('y_true', 'size'),
        n_zeros=('is_zero', 'sum'),
        mean_actual=('y_true', 'mean'),
        mean_pred=('y_pred', 'mean'),
        min_pred=('y_pred', 'min'),
        max_pred=('y_pred', 'max')
    ).reset_index()
    
    decile_stats['n_nonzeros'] = decile_stats['count'] - decile_stats['n_zeros']
    decile_stats['pct_zeros'] = 100 * decile_stats['n_zeros'] / decile_stats['count']
    
    print("\n" + "="*70)
    print(f"{model_name} - DECILE BREAKDOWN (Zero vs Non-Zero)")
    print("="*70)
    print(decile_stats.to_string(index=False))
    
    # Check for perfect separation
    low_deciles_all_zero = (decile_stats[decile_stats['decile'] <= 8]['pct_zeros'] == 100).all()
    high_deciles_no_zero = (decile_stats[decile_stats['decile'] >= 9]['pct_zeros'] < 10).all()
    
    if low_deciles_all_zero and high_deciles_no_zero:
        print("\n🚨 CRITICAL: Perfect zero/non-zero separation detected!")
        print("   This suggests DATA LEAKAGE - a feature is revealing claim occurrence.")
    elif low_deciles_all_zero:
        print("\n⚠️  WARNING: Low deciles are ALL zeros - possible leakage.")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Stacked bar chart
    x = decile_stats['decile']
    axes[0].bar(x, decile_stats['n_zeros'], label='Zeros', color='lightgray')
    axes[0].bar(x, decile_stats['n_nonzeros'], bottom=decile_stats['n_zeros'], 
                label='Non-zeros', color='steelblue')
    axes[0].set_xlabel('Decile')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'{model_name} - Zero vs Non-Zero by Decile')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Percentage zeros
    axes[1].bar(x, decile_stats['pct_zeros'], color='coral')
    axes[1].set_xlabel('Decile')
    axes[1].set_ylabel('% Zeros')
    axes[1].set_title(f'{model_name} - Percentage of Zeros by Decile')
    axes[1].axhline(y=50, color='red', linestyle='--', label='50% threshold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return decile_stats

# Analyze all 4 models
decile_m1 = analyze_deciles(y_train.values, preds_m1, "Model 1 (Ordinal)", bins=10)
decile_m2 = analyze_deciles(y_train.values, preds_m2, "Model 2 (Binary)", bins=10)
decile_m3 = analyze_deciles(y_train.values, preds_m3, "Model 3 (Actuarial)", bins=10)
decile_m4 = analyze_deciles(y_train.values, preds_m4, "Model 4 (Custom)", bins=10)

---
## 7 · Recommendations

Based on the diagnostics above, identify issues and suggest fixes.

In [ ]:
print("="*70)
print("DIAGNOSTIC SUMMARY & RECOMMENDATIONS")
print("="*70)

print("\n📊 KEY FINDINGS:")
print(f"  1. Target has {pct_zeros:.1f}% zeros (expected in insurance)")
print(f"  2. Top feature importance (Model 1): {imp1.iloc[0]['importance']:.3f}")
print(f"  3. Top feature importance (Model 2): {imp2.iloc[0]['importance']:.3f}")

print("\n🔍 POTENTIAL ISSUES:")
print("  [ ] Check if top features contain claim/exposure information (LEAKAGE)")
print("  [ ] If deciles 1-8 are 100% zeros, model is only learning 'has claim' indicator")
print("  [ ] If predictions are clustered at 2 values, not continuous → LEAKAGE")
print("  [ ] Check if any exposure columns (ee_*) leaked into features")

print("\n✅ RECOMMENDED ACTIONS:")
print("  1. Review top 10 features - do any reveal claim occurrence?")
print("  2. Check config/column_exclusions.csv - ensure all exposure cols excluded")
print("  3. If leakage found: add feature to column_exclusions.csv and retrain")
print("  4. Consider using TWO-STAGE model:")
print("     - Stage 1: Classification (claim yes/no)")
print("     - Stage 2: Regression (claim amount | claim=yes)")
print("  5. Alternative: Use Tweedie with proper variance power (1.5-1.9)")

print("\n📋 NEXT STEPS:")
print("  1. Identify leaker feature from section 4 (Feature Importance)")
print("  2. Add to config/column_exclusions.csv")
print("  3. Re-run main_train_only.ipynb")
print("  4. Check if lift chart improves (more gradual slope)")

print("="*70)